In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

In [2]:
from src.data.loader import load_session
from src.analytics.driver_analysis import driver_summary
from src.analytics.driver_analysis import sector_analysis

In [3]:
session = load_session(
    2024,
    "Monaco",
    "R"
)

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']


In [4]:
summary = driver_summary(session)

In [5]:
summary

,Driver,Team,Average Lap Time (s),Fastest Lap Time (s),Median Lap Time (s),Lap Time Std Dev (s),Total Valid Laps,Pit Stops,Start Position,Finish Position
0,LEC,Ferrari,109.321,75.162,78.506,270.991,77,1,1.0,1.0
1,PIA,McLaren,109.436,76.281,78.523,271.256,77,1,2.0,2.0
2,SAI,Ferrari,109.487,74.726,78.568,271.710,77,1,3.0,3.0
3,NOR,McLaren,109.565,75.742,78.403,272.315,77,1,4.0,4.0
4,RUS,Mercedes,109.697,75.228,78.651,272.977,77,1,5.0,5.0
5,VER,Red Bull Racing,109.766,74.569,78.574,273.561,77,2,6.0,6.0
6,HAM,Mercedes,109.840,74.165,78.587,274.116,77,2,7.0,7.0
7,TSU,RB,112.143,74.720,80.184,274.505,77,1,8.0,8.0
8,ALB,Williams,112.333,77.060,80.245,275.178,77,1,9.0,9.0
9,GAS,Alpine,112.413,75.625,80.237,275.493,77,1,10.0,10.0


In [6]:
summary.shape

(16, 10)

In [7]:
len(session.results)

20

In [8]:
laps = session.laps.copy()

laps = laps[
    laps["LapTime"].notna() &
    laps["IsAccurate"]
]

print(laps["Driver"].nunique())

16


In [9]:
sorted(laps["Driver"].unique())

['ALB',
 'ALO',
 'BOT',
 'GAS',
 'HAM',
 'LEC',
 'NOR',
 'PIA',
 'RIC',
 'RUS',
 'SAI',
 'SAR',
 'STR',
 'TSU',
 'VER',
 'ZHO']

In [10]:
sorted(session.results["Abbreviation"].tolist())

['ALB',
 'ALO',
 'BOT',
 'GAS',
 'HAM',
 'HUL',
 'LEC',
 'MAG',
 'NOR',
 'OCO',
 'PER',
 'PIA',
 'RIC',
 'RUS',
 'SAI',
 'SAR',
 'STR',
 'TSU',
 'VER',
 'ZHO']

In [11]:
for driver in ["HUL", "MAG", "OCO", "PER"]:
    print("=" * 30)
    print(driver)
    display(
        session.laps.pick_drivers(driver)[
            ["LapNumber", "LapTime", "IsAccurate", "Deleted"]
        ].head(10)
    )

HUL


,LapNumber,LapTime,IsAccurate,Deleted
1235,1.0,NaT,False,False


MAG


,LapNumber,LapTime,IsAccurate,Deleted
1236,1.0,NaT,False,False


OCO


,LapNumber,LapTime,IsAccurate,Deleted
1233,1.0,NaT,False,False


PER


,LapNumber,LapTime,IsAccurate,Deleted
1234,1.0,NaT,False,False


In [12]:
sector_df = sector_analysis(session)

In [13]:
sector_df

,Driver,Team,Average Sector 1 (s),Average Sector 2 (s),Average Sector 3 (s)
0,ALB,Williams,21.346,37.626,22.012
1,ALO,Aston Martin,21.306,37.855,21.989
2,BOT,Kick Sauber,21.271,37.749,22.025
3,GAS,Alpine,21.203,37.585,22.240
4,HAM,Mercedes,20.841,36.603,21.160
5,LEC,Ferrari,20.548,36.857,21.034
6,NOR,McLaren,20.574,36.964,20.995
7,PIA,McLaren,20.530,36.967,21.027
8,RIC,RB,21.339,37.799,22.092
9,RUS,Mercedes,20.882,36.744,20.963


In [14]:
sector_df.shape

(16, 5)

In [15]:
sector_df.loc[sector_df["Driver"] == "NOR"]

,Driver,Team,Average Sector 1 (s),Average Sector 2 (s),Average Sector 3 (s)
6,NOR,McLaren,20.574,36.964,20.995


In [16]:
laps = session.laps.pick_drivers("NOR")

laps = laps.dropna(
    subset=[
        "Sector1Time",
        "Sector2Time",
        "Sector3Time",
    ]
)

laps["Sector1Time"].dt.total_seconds().mean()

np.float64(20.57380263157895)

In [17]:
from importlib import reload
import src.analytics.driver_analysis as da

reload(da)

summary = da.driver_summary(session)
summary.head()

sector = da.sector_analysis(session)
sector.head()

,Driver,Team,Average Sector 1 (s),Average Sector 2 (s),Average Sector 3 (s)
0,ALB,Williams,21.346,37.626,22.012
1,ALO,Aston Martin,21.306,37.855,21.989
2,BOT,Kick Sauber,21.271,37.749,22.025
3,GAS,Alpine,21.203,37.585,22.240
4,HAM,Mercedes,20.841,36.603,21.160


In [18]:
from importlib import reload
import src.analytics.driver_analysis as da

reload(da)

<module 'src.analytics.driver_analysis' from 'E:\\devansh\\F1-Race-Intelligence\\src\\analytics\\driver_analysis.py'>

In [19]:
speed = da.speed_analysis(session)
speed

,Driver,Team,Average Speed I1 (km/h),Average Speed I2 (km/h),Average Finish Line Speed (km/h),Average Speed Trap (km/h),Maximum Speed Trap (km/h)
0,ALB,Williams,185.10,175.31,255.59,276.56,283.0
1,ALO,Aston Martin,198.24,172.62,255.20,274.49,281.0
2,BOT,Kick Sauber,185.41,178.45,263.16,277.57,286.0
3,GAS,Alpine,189.08,177.47,256.11,274.99,281.0
4,HAM,Mercedes,193.55,186.34,259.69,277.23,289.0
5,LEC,Ferrari,199.17,182.42,255.39,275.57,282.0
6,NOR,McLaren,193.30,185.74,263.00,279.30,285.0
7,PIA,McLaren,194.69,186.21,259.57,278.14,283.0
8,RIC,RB,192.47,174.78,260.68,272.62,284.0
9,RUS,Mercedes,196.66,181.81,259.03,272.71,285.0


In [20]:
speed.sort_values(
    "Maximum Speed Trap (km/h)",
    ascending=False
)

,Driver,Team,Average Speed I1 (km/h),Average Speed I2 (km/h),Average Finish Line Speed (km/h),Average Speed Trap (km/h),Maximum Speed Trap (km/h)
4,HAM,Mercedes,193.55,186.34,259.69,277.23,289.0
11,SAR,Williams,193.71,174.74,259.23,277.93,288.0
10,SAI,Ferrari,194.95,185.04,260.33,277.34,288.0
15,ZHO,Kick Sauber,182.78,178.99,261.07,277.11,287.0
14,VER,Red Bull Racing,187.22,186.36,259.52,268.78,286.0
2,BOT,Kick Sauber,185.41,178.45,263.16,277.57,286.0
6,NOR,McLaren,193.30,185.74,263.00,279.30,285.0
12,STR,Aston Martin,185.00,180.22,259.95,275.01,285.0
9,RUS,Mercedes,196.66,181.81,259.03,272.71,285.0
8,RIC,RB,192.47,174.78,260.68,272.62,284.0


In [21]:
from importlib import reload
import src.analytics.driver_analysis as da

reload(da)

<module 'src.analytics.driver_analysis' from 'E:\\devansh\\F1-Race-Intelligence\\src\\analytics\\driver_analysis.py'>

In [22]:
report = da.driver_report(session, "HAM")

In [23]:
report

Driver                                   HAM
Team                                Mercedes
Average Lap Time (s)                  109.84
Fastest Lap Time (s)                  74.165
Median Lap Time (s)                   78.587
Lap Time Std Dev (s)                 274.116
Total Valid Laps                          77
Pit Stops                                  2
Start Position                           7.0
Finish Position                          7.0
Average Sector 1 (s)                  20.841
Average Sector 2 (s)                  36.603
Average Sector 3 (s)                   21.16
Average Speed I1 (km/h)               193.55
Average Speed I2 (km/h)               186.34
Average Finish Line Speed (km/h)      259.69
Average Speed Trap (km/h)             277.23
Maximum Speed Trap (km/h)              289.0
Name: 6, dtype: object

In [24]:
report_lec = da.driver_report(session, "LEC")
report_lec

Driver                                  LEC
Team                                Ferrari
Average Lap Time (s)                109.321
Fastest Lap Time (s)                 75.162
Median Lap Time (s)                  78.506
Lap Time Std Dev (s)                270.991
Total Valid Laps                         77
Pit Stops                                 1
Start Position                          1.0
Finish Position                         1.0
Average Sector 1 (s)                 20.548
Average Sector 2 (s)                 36.857
Average Sector 3 (s)                 21.034
Average Speed I1 (km/h)              199.17
Average Speed I2 (km/h)              182.42
Average Finish Line Speed (km/h)     255.39
Average Speed Trap (km/h)            275.57
Maximum Speed Trap (km/h)             282.0
Name: 0, dtype: object

In [25]:
report_invalid = da.driver_report(session, "XYZ")
report_invalid

ValueError: No data found for driver 'XYZ'.

In [26]:
from importlib import reload
import src.analytics.driver_analysis as da

reload(da)

positions = da.position_changes(session)
positions

,Driver,Team,Start Position,Finish Position,Positions Gained
0,BOT,Kick Sauber,17.0,13.0,4.0
1,ALO,Aston Martin,14.0,11.0,3.0
2,ZHO,Kick Sauber,18.0,16.0,2.0
3,LEC,Ferrari,1.0,1.0,0.0
4,RUS,Mercedes,5.0,5.0,0.0
5,PIA,McLaren,2.0,2.0,0.0
6,SAI,Ferrari,3.0,3.0,0.0
7,NOR,McLaren,4.0,4.0,0.0
8,TSU,RB,8.0,8.0,0.0
9,HAM,Mercedes,7.0,7.0,0.0


In [27]:
positions.loc[
    positions["Driver"] == "NOR"
]

,Driver,Team,Start Position,Finish Position,Positions Gained
7,NOR,McLaren,4.0,4.0,0.0


In [28]:
from importlib import reload
import src.analytics.driver_analysis as da

reload(da)

<module 'src.analytics.driver_analysis' from 'E:\\devansh\\F1-Race-Intelligence\\src\\analytics\\driver_analysis.py'>

In [30]:
tyres = da.tyre_usage(session)

tyres

,Driver,Team,Compounds Used,Number of Stints,Average Tyre Life,Maximum Tyre Life,Started on Fresh Tyres
0,ALB,Williams,"HARD, MEDIUM",2,38.01,76.0,True
1,ALO,Aston Martin,"HARD, MEDIUM",2,38.51,76.0,False
2,BOT,Kick Sauber,"HARD, MEDIUM",3,27.08,62.0,True
3,GAS,Alpine,"HARD, MEDIUM",2,38.01,76.0,True
4,HAM,Mercedes,"HARD, MEDIUM",3,21.82,50.0,True
5,LEC,Ferrari,"HARD, MEDIUM",2,39.00,77.0,True
6,NOR,McLaren,"HARD, MEDIUM",2,40.00,78.0,False
7,PIA,McLaren,"HARD, MEDIUM",2,40.00,78.0,False
8,RIC,RB,"HARD, MEDIUM",2,38.50,76.0,True
9,RUS,Mercedes,"HARD, MEDIUM",2,39.00,77.0,True
